# 02. Train FFT Dictionary & Classifier
**Objective:** Train the Dictionary Learning algorithm on pure shift-invariant frequency data, extract sparse codes, and train the final inference classifier.

In [5]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

sys.path.append(os.path.abspath('../'))
from src.config import PREPROCESSED_DIR, MODELS_DIR, MOVEMENT_LABELS
from src.features import extract_fft_magnitude, EMGDictionaryLearner

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1. Load the Dense Dataset

In [6]:
data_path = os.path.join(PREPROCESSED_DIR, "DB1_S1_Dense.h5")
with h5py.File(data_path, 'r') as f:
    X_bal = np.array(f['X'])
    y_bal = np.array(f['y']).astype(np.int64)
    reps_bal = np.array(f['reps'])

# Standard NinaPro DB1 Train/Test Split (Reps 2, 5 for testing)
train_reps = [1, 3, 4, 6, 7, 8, 9, 10] # Adjusting to handle up to 10 reps if applicable, though paper uses 2 and 5 for test
test_reps = [2, 5]

train_idx = np.where(np.isin(reps_bal, train_reps))[0]
test_idx = np.where(np.isin(reps_bal, test_reps))[0]

X_train, y_train = X_bal[train_idx], y_bal[train_idx]
X_test, y_test = X_bal[test_idx], y_bal[test_idx]

print(f"Training Windows: {X_train.shape[0]}")
print(f"Testing Windows: {X_test.shape[0]}")

Training Windows: 143956
Testing Windows: 37562


### 2. Extract Shift-Invariant Frequency Features

In [7]:
X_train_freq = extract_fft_magnitude(X_train)
X_test_freq = extract_fft_magnitude(X_test)

print(f"FFT Training Shape: {X_train_freq.shape}")

Extracting FFT magnitudes...
Extracting FFT magnitudes...
FFT Training Shape: (143956, 11, 10)


### 3. Train Dictionary & Extract Sparse Codes

In [8]:
# We use 128 atoms to build a rich physical vocabulary of muscle frequencies
dl_model = EMGDictionaryLearner(n_atoms=128, n_nonzero_coefs=5)
dl_model.fit(X_train_freq)

# Save the trained dictionary for the final prediction script
dl_model.save(os.path.join(MODELS_DIR, "fft_dictionary_s1.pkl"))

print("\nTransforming FFT data into Sparse Codes...")
X_train_sparse = dl_model.transform(X_train_freq)
X_test_sparse = dl_model.transform(X_test_freq)

print(f"Sparse Feature Shape: {X_train_sparse.shape}")

Fitting Dictionary (128 atoms) on shape (143956, 110)...
Dictionary learning complete.

Transforming FFT data into Sparse Codes...
Sparse Feature Shape: (143956, 128)


### 4. Train the Final Machine Learning Classifier

In [9]:
# Standardize the sparse codes for the SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sparse)
X_test_scaled = scaler.transform(X_test_sparse)

# Save the scaler for inference
joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler_s1.pkl"))

print("\nTraining Linear SVM on Sparse Codes...")
classifier = LinearSVC(C=1.0, dual=False, max_iter=1000)
classifier.fit(X_train_scaled, y_train)

# Save the classifier for inference
joblib.dump(classifier, os.path.join(MODELS_DIR, "svm_classifier_s1.pkl"))


Training Linear SVM on Sparse Codes...


['/workspaces/TCC/models/svm_classifier_s1.pkl']

### 5. Evaluate and Test the Lexicon Translation

In [10]:
preds = classifier.predict(X_test_scaled)
acc = accuracy_score(y_test, preds)

print(f"\n--- Final Model Accuracy ---")
print(f"Accuracy: {acc * 100:.2f}%\n")

print("--- Lexicon Translation Test ---")
# Pick 3 random test samples to prove the text translation works
np.random.seed(42)
sample_indices = np.random.choice(len(preds), 3, replace=False)

for idx in sample_indices:
    true_label = y_test[idx]
    pred_label = preds[idx]
    
    true_text = MOVEMENT_LABELS[true_label]
    pred_text = MOVEMENT_LABELS[pred_label]
    
    print(f"True: [{true_label:02d}] {true_text}")
    print(f"Pred: [{pred_label:02d}] {pred_text}")
    print("-" * 40)


--- Final Model Accuracy ---
Accuracy: 34.31%

--- Lexicon Translation Test ---
True: [01] Index flexion
Pred: [12] Thumb extension
----------------------------------------
True: [01] Index flexion
Pred: [12] Thumb extension
----------------------------------------
True: [19] Pointing index
Pred: [19] Pointing index
----------------------------------------
